# 📓 GOOGLE COLAB NOTEBOOK

## JSON → Corporate PPT Generator

## 🟦 CELL 1 — Install Dependencies

In [ ]:
%pip install python-pptx pydantic

## 🟦 CELL 2 — Imports & Constants

In [ ]:
from pptx import Presentation
from pptx.util import Inches, Pt
from pptx.enum.text import PP_ALIGN, MSO_ANCHOR
from pptx.dml.color import RGBColor
from pptx.enum.shapes import MSO_SHAPE
from pptx.chart.data import CategoryChartData
from pptx.enum.chart import XL_CHART_TYPE, XL_LEGEND_POSITION
import json
import os
from typing import List, Optional, Union, Literal, Dict, Any
from pydantic import BaseModel, Field, ValidationError, validator
from IPython.display import display, HTML

## 🟦 CELL 3 — Theme Definition (LOCKED)

> Single theme used for **all slides**

In [ ]:
# ==========================================
# 🟦 DATA MODELS (STRICT VALIDATION)
# ==========================================
# These models ensure your JSON is correct before generating the PPT.
# They also define the structure for the "Theme" and "Slides".

class ThemeColors(BaseModel):
    background: List[int] = [255, 255, 255]
    title: List[int] = [47, 93, 140]
    text: List[int] = [31, 41, 51]
    accent: List[int] = [74, 144, 226]
    border: List[int] = [217, 221, 225]
    table_header: List[int] = [47, 93, 140]
    
class ThemeFonts(BaseModel):
    family: str = "Calibri"
    size_title: int = 36
    size_body: int = 18

class PresentationTheme(BaseModel):
    colors: ThemeColors = ThemeColors()
    fonts: ThemeFonts = ThemeFonts()

# --- Slide Content Models ---

class SlideCover(BaseModel):
    type: Literal["cover"]
    title: str
    subtitle: Optional[str] = ""

class SlideContent(BaseModel):
    type: Literal["content"]
    title: str
    banner: Optional[str] = None
    text: str

class TableColumn(BaseModel):
    key: str
    label: str

class SlideTable(BaseModel):
    type: Literal["table"]
    title: str
    columns: List[TableColumn]
    rows: List[Dict[str, Any]]

class ChartData(BaseModel):
    categories: List[str]
    series_name: str
    values: List[float]

class SlideChart(BaseModel):
    type: Literal["chart"]
    title: str
    chart: ChartData
    chart_type: Literal["COLUMN", "BAR", "LINE"] = "COLUMN"

class CardItem(BaseModel):
    title: str
    items: List[str]

class SlideCards(BaseModel):
    type: Literal["cards"]
    title: str
    cards: List[CardItem]

class GanttTask(BaseModel):
    name: str
    start_week: int
    duration: int
    progress: int = 0

class SlideGantt(BaseModel):
    type: Literal["gantt"]
    title: str
    tasks: List[GanttTask]

class SlideImage(BaseModel):
    type: Literal["image"]
    title: str
    image_path: str
    caption: Optional[str] = ""

# Union of all possible slide types
SlideType = Union[
    SlideCover, SlideContent, SlideTable, SlideChart, 
    SlideCards, SlideGantt, SlideImage
]

class PresentationMeta(BaseModel):
    title: str
    author: str
    date: str

class PresentationConfig(BaseModel):
    meta: PresentationMeta
    theme: PresentationTheme = PresentationTheme()
    slides: List[SlideType]

print("✅ Data Models Defined. Your JSON will now be strictly checked!")

## 🟦 CELL 4 — PPT Helper Functions

In [ ]:
# ==========================================
# 🟦 RENDERER HELPERS
# ==========================================

def get_rgb(color_list):
    return RGBColor(color_list[0], color_list[1], color_list[2])

def add_common_elements(slide, title, theme: PresentationTheme):
    """Adds the title and footer to every slide."""
    # Title
    title_shape = slide.shapes.add_textbox(Inches(0.5), Inches(0.3), Inches(12.33), Inches(1))
    tf = title_shape.text_frame
    p = tf.paragraphs[0]
    p.text = title
    p.font.size = Pt(theme.fonts.size_title)
    p.font.bold = True
    p.font.color.rgb = get_rgb(theme.colors.title)
    p.font.name = theme.fonts.family
    
    # Decorative Line (Underline)
    line = slide.shapes.add_shape(
        MSO_SHAPE.RECTANGLE, Inches(0.5), Inches(1.1), Inches(12.33), Inches(0.03)
    )
    line.fill.solid()
    line.fill.fore_color.rgb = get_rgb(theme.colors.border) # Subtle grey line
    line.line.fill.background()

def render_cover(slide, data: SlideCover, theme: PresentationTheme):
    # Clean White Background
    bg = slide.shapes.add_shape(MSO_SHAPE.RECTANGLE, Inches(0), Inches(0), Inches(13.33), Inches(7.5))
    bg.fill.solid()
    bg.fill.fore_color.rgb = get_rgb(theme.colors.background)
    
    # Title (Left Aligned as per screenshot)
    tb = slide.shapes.add_textbox(Inches(1.0), Inches(2.5), Inches(11.33), Inches(2.0))
    p = tb.text_frame.paragraphs[0]
    p.text = data.title
    p.font.size = Pt(48)
    p.font.bold = True
    p.font.color.rgb = RGBColor(0, 0, 0) # Black Title
    p.font.name = "Times New Roman" # Serif font as per screenshot
    p.alignment = PP_ALIGN.LEFT
    
    # Subtitle
    if data.subtitle:
        sb = slide.shapes.add_textbox(Inches(1.0), Inches(4.5), Inches(11.33), Inches(1.5))
        p = sb.text_frame.paragraphs[0]
        p.text = data.subtitle
        p.font.size = Pt(18)
        p.font.color.rgb = get_rgb(theme.colors.text)
        p.alignment = PP_ALIGN.LEFT

def render_content(slide, data: SlideContent, theme: PresentationTheme):
    top_cursor = 1.5
    
    if data.banner:
        # Light Blue Banner Box
        box = slide.shapes.add_shape(MSO_SHAPE.RECTANGLE, Inches(0.5), Inches(top_cursor), Inches(12.33), Inches(1.0))
        box.fill.solid()
        box.fill.fore_color.rgb = RGBColor(220, 230, 241) # Very light blue
        box.line.color.rgb = RGBColor(220, 230, 241)
        
        tf = box.text_frame
        tf.margin_left = Inches(0.2)
        p = tf.paragraphs[0]
        p.text = data.banner
        p.font.size = Pt(14)
        p.font.color.rgb = RGBColor(0, 0, 0)
        p.alignment = PP_ALIGN.LEFT
        top_cursor += 1.2

    # Main Text
    tb = slide.shapes.add_textbox(Inches(0.5), Inches(top_cursor), Inches(12.33), Inches(5))
    tf = tb.text_frame
    tf.word_wrap = True
    p = tf.paragraphs[0]
    p.text = data.text
    p.font.size = Pt(theme.fonts.size_body)
    p.font.color.rgb = get_rgb(theme.colors.text)
    p.font.name = theme.fonts.family

def render_table(slide, data: SlideTable, theme: PresentationTheme):
    rows = len(data.rows) + 1
    cols = len(data.columns)
    
    # Dynamic Width Calculation
    table_width = Inches(12.33)
    col_width = table_width / cols
    
    shape = slide.shapes.add_table(rows, cols, Inches(0.5), Inches(2.0), table_width, Inches(0.6 * rows))
    table = shape.table
    
    # Headers
    for i, col in enumerate(data.columns):
        cell = table.cell(0, i)
        cell.text = col.label
        cell.fill.solid()
        cell.fill.fore_color.rgb = RGBColor(245, 245, 245) # Light Grey
        p = cell.text_frame.paragraphs[0]
        p.font.bold = True
        p.font.color.rgb = RGBColor(0, 0, 0)
        p.alignment = PP_ALIGN.LEFT
        
    # Rows
    for r, row_data in enumerate(data.rows, start=1):
        for c, col in enumerate(data.columns):
            cell = table.cell(r, c)
            val = str(row_data.get(col.key, ""))
            cell.text = val
            p = cell.text_frame.paragraphs[0]
            p.font.size = Pt(12)
            p.font.color.rgb = get_rgb(theme.colors.text)
            p.alignment = PP_ALIGN.LEFT
            
            # Status Coloring
            val_lower = val.lower()
            if col.key == "status":
                if "completed" in val_lower:
                    p.font.color.rgb = RGBColor(0, 176, 80) # Green
                    cell.text = "✅ " + val
                elif "in progress" in val_lower:
                    p.font.color.rgb = RGBColor(0, 112, 192) # Blue
                    cell.text = "🔄 " + val
                elif "pending" in val_lower:
                    p.font.color.rgb = RGBColor(255, 0, 0) # Red
                    cell.text = "⚠️ " + val

def render_chart(slide, data: SlideChart, theme: PresentationTheme):
    chart_data = CategoryChartData()
    chart_data.categories = data.chart.categories
    chart_data.add_series(data.chart.series_name, data.chart.values)
    
    chart_type = XL_CHART_TYPE.COLUMN_CLUSTERED
    if data.chart_type == "LINE": chart_type = XL_CHART_TYPE.LINE
    if data.chart_type == "BAR": chart_type = XL_CHART_TYPE.BAR_CLUSTERED
    
    chart = slide.shapes.add_chart(
        chart_type, Inches(0.5), Inches(2.0), Inches(12.33), Inches(4.5), chart_data
    ).chart
    
    chart.has_legend = True
    chart.legend.position = XL_LEGEND_POSITION.BOTTOM

def render_cards(slide, data: SlideCards, theme: PresentationTheme):
    count = len(data.cards)
    if count == 0: return
    
    margin = 0.5
    spacing = 0.3
    available_w = 13.33 - (2 * margin) - ((count - 1) * spacing)
    card_w = available_w / count
    
    for i, card in enumerate(data.cards):
        left = margin + (i * (card_w + spacing))
        top = 2.0
        
        # Card Box
        box = slide.shapes.add_shape(MSO_SHAPE.ROUNDED_RECTANGLE, Inches(left), Inches(top), Inches(card_w), Inches(4.5))
        box.fill.solid()
        box.fill.fore_color.rgb = RGBColor(255, 255, 255)
        box.line.color.rgb = get_rgb(theme.colors.border)
        
        # Header Strip
        header = slide.shapes.add_shape(MSO_SHAPE.RECTANGLE, Inches(left), Inches(top), Inches(card_w), Inches(0.8))
        header.fill.solid()
        header.fill.fore_color.rgb = get_rgb(theme.colors.title) # Blue header
        
        # Title
        tf = header.text_frame
        p = tf.paragraphs[0]
        p.text = card.title
        p.font.bold = True
        p.font.color.rgb = RGBColor(255, 255, 255)
        p.alignment = PP_ALIGN.CENTER
        
        # Items
        content_box = slide.shapes.add_textbox(Inches(left + 0.1), Inches(top + 0.9), Inches(card_w - 0.2), Inches(3.5))
        for item in card.items:
            p = content_box.text_frame.add_paragraph()
            p.text = f"• {item}"
            p.font.size = Pt(14)
            p.font.color.rgb = get_rgb(theme.colors.text)
            p.space_after = Pt(10)

def render_gantt(slide, data: SlideGantt, theme: PresentationTheme):
    tasks = data.tasks
    if not tasks: return
    
    start_x = 3.0
    start_y = 2.5
    row_h = 0.6
    chart_w = 9.5
    
    min_w = min(t.start_week for t in tasks)
    max_w = max(t.start_week + t.duration for t in tasks)
    total_w = max_w - min_w + 1
    week_w = chart_w / total_w
    
    # Header
    for w in range(total_w):
        x = start_x + (w * week_w)
        lbl = slide.shapes.add_textbox(Inches(x), Inches(start_y - 0.4), Inches(week_w), Inches(0.4))
        p = lbl.text_frame.paragraphs[0]
        p.text = f"W{min_w + w}"
        p.font.size = Pt(10)
        p.alignment = PP_ALIGN.CENTER
        
        # Gridline
        line = slide.shapes.add_shape(MSO_SHAPE.LINE_INVERSE, Inches(x), Inches(start_y), Inches(0), Inches(len(tasks)*row_h))
        line.line.color.rgb = get_rgb(theme.colors.border)
        line.line.dash_style = 1

    # Tasks
    for i, task in enumerate(tasks):
        y = start_y + (i * row_h)
        
        # Label
        tb = slide.shapes.add_textbox(Inches(0.5), Inches(y), Inches(2.4), Inches(row_h))
        p = tb.text_frame.paragraphs[0]
        p.text = task.name
        p.font.size = Pt(12)
        p.alignment = PP_ALIGN.RIGHT
        
        # Bar
        bar_x = start_x + ((task.start_week - min_w) * week_w)
        bar_w = task.duration * week_w
        bar = slide.shapes.add_shape(MSO_SHAPE.ROUNDED_RECTANGLE, Inches(bar_x), Inches(y+0.1), Inches(bar_w), Inches(row_h-0.2))
        bar.fill.solid()
        bar.fill.fore_color.rgb = get_rgb(theme.colors.accent)
        
        if task.progress > 0:
            p = bar.text_frame.paragraphs[0]
            p.text = f"{task.progress}%"
            p.font.size = Pt(9)
            p.font.color.rgb = RGBColor(255, 255, 255)
            p.alignment = PP_ALIGN.CENTER

def render_image(slide, data: SlideImage, theme: PresentationTheme):
    if os.path.exists(data.image_path):
        pic = slide.shapes.add_picture(data.image_path, Inches(1), Inches(2.0), height=Inches(4.5))
        pic.left = int((Inches(13.33) - pic.width) / 2)
    else:
        tb = slide.shapes.add_textbox(Inches(1), Inches(3), Inches(11), Inches(2))
        tb.text_frame.text = f"Image not found: {data.image_path}"
        
    if data.caption:
        tb = slide.shapes.add_textbox(Inches(1), Inches(6.6), Inches(11.33), Inches(0.5))
        p = tb.text_frame.paragraphs[0]
        p.text = data.caption
        p.font.italic = True
        p.alignment = PP_ALIGN.CENTER

## 🟦 CELL 5 — Table Renderer

## 🟦 CELL 6 — Card Renderer

## 🟦 CELL 7 — Timeline Renderer

## 🟦 CELL 8 — FULL JSON INPUT (Random Project Proposal)

> **ONLY this JSON controls the PPT**

In [ ]:
# ==========================================
# 🟦 MASTER JSON INPUT (COPIED FROM SCREENSHOTS)
# ==========================================

raw_json_input = {
  "meta": {
    "title": "Project Report",
    "author": "Supreme Ventures Group",
    "date": "Dec 2025"
  },
  "theme": {
    "colors": {
      "background": [255, 255, 255],
      "title": [0, 0, 0],             # Black Titles
      "text": [50, 50, 50],           # Dark Gray Text
      "accent": [0, 112, 192],        # Blue Accent
      "table_header": [245, 245, 245],# Light Grey Header
      "border": [200, 200, 200]
    },
    "fonts": {
      "family": "Arial",
      "size_title": 36,
      "size_body": 14
    }
  },
  "slides": [
    # --- SLIDE 1: COVER (Matches Screenshot) ---
    { 
      "type": "cover", 
      "title": "Project Report - Biller Management Portal", 
      "subtitle": "Bill Payment Platform Enhancement to support Standardized Onboarding of Small and Medium Sized Billers\nVersion 1.1 | December 11, 2025" 
    },

    # --- SLIDE 2: TIMELINE (Matches Screenshot) ---
    { 
      "type": "table", 
      "title": "Project Timeline & Current Status", 
      "columns": [ 
        { "key": "phase", "label": "Phase" }, 
        { "key": "timeline", "label": "Timeline" }, 
        { "key": "status", "label": "Status" },
        { "key": "notes", "label": "Notes" }
      ], 
      "rows": [
        { "phase": "Requirements & UI Design", "timeline": "29th Sep - 12th Oct 2025", "status": "Completed", "notes": "All core UI/UX designs are completed" },
        { "phase": "Onboarding Development", "timeline": "6th Oct - 16 Nov 2025", "status": "Completed", "notes": "Multi-step admin onboarding wizard fully functional" },
        { "phase": "Desktop/mPOS Implementation", "timeline": "13th Oct - 30th Nov 2025", "status": "Completed", "notes": "Admin and Biller self-service web portals" },
        { "phase": "CMS Platform", "timeline": "27th Oct - 7th Dec 2025", "status": "Completed", "notes": "Admin Portal - Complete biller and user management" },
        { "phase": "Reporting - Billers", "timeline": "24th Nov - 8th Dec 2025", "status": "Completed", "notes": "Customer & Bill History, Master Bill Payment Report" },
        { "phase": "Bulk and Custom Exports", "timeline": "24th Nov - 8th Dec 2025", "status": "Completed", "notes": "CSV import/export built" },
        { "phase": "User Acceptance Testing", "timeline": "1st Dec - 8th Dec 2025", "status": "In Progress", "notes": "Awaiting stakeholder review" }
      ]
    },

    # --- SLIDE 3: SCOPE (Dummy Content based on context) ---
    { 
      "type": "cards", 
      "title": "Project Scope", 
      "cards": [
        { "title": "Sports Book Integration", "items": ["Integration with Altenar Sports Book", "Seamless wallet integration APIs"] },
        { "title": "Desktop and mPOS", "items": ["Interface for Bet Shops", "Secure retail setup for transactions"] },
        { "title": "Online Consumer Platform", "items": ["Single sign-on interface", "Top-up and withdraw mechanisms"] }
      ]
    },

    # --- SLIDE 4: ARCHITECTURE (Dummy Image) ---
    {
      "type": "image",
      "title": "System Architecture",
      "image_path": "dummy_architecture.png",
      "caption": "Figure 1: High-Level System Architecture"
    },

    # --- SLIDE 5: FINANCIALS (Dummy Data) ---
    { 
      "type": "chart", 
      "title": "Budget Utilization", 
      "chart_type": "COLUMN", 
      "chart": { 
        "categories": ["Design", "Dev", "QA", "Infra"], 
        "series_name": "Spend ($k)", 
        "values": [45, 120, 30, 15] 
      } 
    }
  ]
}
print("✅ JSON Loaded with Screenshot Content.")

## 🟦 CELL 9 — PPT Generator (JSON → PPT)

In [ ]:
# ==========================================
# 🟦 MAIN GENERATOR (WITH VALIDATION)
# ==========================================

try:
    # 1. Validate JSON against Pydantic Models
    print("🔍 Validating JSON Structure...")
    config = PresentationConfig(**raw_json_input)
    print("✅ Validation Successful! Generating PPT...")

    # 2. Initialize Presentation
    prs = Presentation()
    prs.slide_width = Inches(13.333)
    prs.slide_height = Inches(7.5)
    
    theme = config.theme

    # 3. Loop through Slides
    for slide_data in config.slides:
        # Create blank slide (Layout 6 is usually blank)
        slide = prs.slides.add_slide(prs.slide_layouts[6])
        
        # Apply Background
        bg = slide.background
        fill = bg.fill
        fill.solid()
        fill.fore_color.rgb = get_rgb(theme.colors.background)

        # Dispatch to Renderers
        if slide_data.type == "cover":
            render_cover(slide, slide_data, theme)
        
        elif slide_data.type == "content":
            add_common_elements(slide, slide_data.title, theme)
            render_content(slide, slide_data, theme)
            
        elif slide_data.type == "table":
            add_common_elements(slide, slide_data.title, theme)
            render_table(slide, slide_data, theme)
            
        elif slide_data.type == "chart":
            add_common_elements(slide, slide_data.title, theme)
            render_chart(slide, slide_data, theme)
            
        elif slide_data.type == "cards":
            add_common_elements(slide, slide_data.title, theme)
            render_cards(slide, slide_data, theme)
            
        elif slide_data.type == "gantt":
            add_common_elements(slide, slide_data.title, theme)
            render_gantt(slide, slide_data, theme)
            
        elif slide_data.type == "image":
            add_common_elements(slide, slide_data.title, theme)
            render_image(slide, slide_data, theme)

    # 4. Save
    output_path = "enterprise_presentation.pptx"
    prs.save(output_path)
    print(f"🎉 Presentation saved to: {output_path}")

    # 5. Download Link
    if os.path.exists(output_path):
        with open(output_path, "rb") as f:
            b64 = base64.b64encode(f.read()).decode()
        
        download_link = f'<a href="data:application/vnd.openxmlformats-officedocument.presentationml.presentation;base64,{b64}" download="{output_path}" style="font-size: 20px; font-weight: bold; color: blue; background-color: #f0f0f0; padding: 10px; border-radius: 5px; text-decoration: none;">⬇️ DOWNLOAD {output_path}</a>'
        display(HTML(download_link))

except ValidationError as e:
    print("❌ JSON VALIDATION ERROR! Please fix your JSON input.")
    print("---------------------------------------------------")
    # Print a friendly error
    for err in e.errors():
        loc = " -> ".join([str(x) for x in err['loc']])
        print(f"Error in [{loc}]: {err['msg']}")
except Exception as e:
    print(f"❌ UNEXPECTED ERROR: {str(e)}")

## 🟦 CELL 10 — Save & Download PPT

In [ ]:
import base64
import os
from IPython.display import HTML, display

output_path = "project_proposal.pptx"
prs.save(output_path)
print(f"Presentation saved to: {output_path}")

if os.path.exists(output_path):
    with open(output_path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode()
    
    # Create a clickable download link
    download_link = f'<a href="data:application/vnd.openxmlformats-officedocument.presentationml.presentation;base64,{b64}" download="{output_path}" style="font-size: 20px; font-weight: bold; color: blue;">⬇️ Click Here to Download PPT</a>'
    display(HTML(download_link))